In [1]:
import requests
import json
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from tqdm import tqdm


In [2]:
# -----------------------------
# 1. Basic Setup
# -----------------------------
API_BASE = "https://api.deepseek.com"
TOKEN = "sk-509adf05adb04b3f902603ee186563c8"
MODEL_NAME = 'deepseek-reasoner' 

headers = {
    'Authorization': f'Bearer {TOKEN}',
    'Content-Type': 'application/json'
}

def query_llm(prompt, model=MODEL_NAME, max_tokens=256, temperature=0.0):
    """
    Sends a prompt to the local LLM and returns the response text.
    :param prompt: string, the full text you want to pass to the LLM
    :param model: string, model name, e.g. "llama3.2:3b"
    :param max_tokens: int, maximum tokens for the generation
    :param temperature: float, sampling temperature
    :return: string, response content from LLM
    """
    url = f"{API_BASE}/api/chat/completions"
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        # Optional generation parameters
        "max_tokens": max_tokens,
        "temperature": temperature
    }
    response = requests.post(url, headers=headers, json=data)
    response_json = response.json()
    # The actual text is typically found in response_json["choices"][0]["message"]["content"]
    try:
        return response_json["choices"][0]["message"]["content"].strip()
    except (KeyError, IndexError):
        return "Error: Unable to parse LLM response."


In [3]:
import pandas as pd

# Load the provided dataset to examine its structure
file_path = "S:/School/Year 2/2024-25c-fai2-adsai-SallyIbrahim211066/models/Merged_Processed_group6.csv"
dataset = pd.read_csv(file_path)
sentences = dataset['Translation'].tolist()
gold_labels = dataset['Emotion'].tolist()

In [ ]:
# -----------------------------
# 2. Baseline Prompt
# -----------------------------
baseline_prompt_template = (
    "You are classifying an English-translated sentence from an Arabic conversation (originally from the show ‘The Blind Date Show 2’). "
    "Emotion cues may be subtle or partially lost in translation. Your task is to carefully detect any relevant emotion and classify "
    "the sentence into exactly one of the following categories:\n\n"
    "1) neutral: purely factual, small talk, or no strong emotion\n"
    "2) happiness: joy, excitement, laughter, or positive/affectionate tone\n"
    "3) surprise: shock, astonishment, or unexpected reaction (e.g., ‘wow’, ‘didn't expect’)\n"
    "4) sadness: sorrow, disappointment, or negative reflection (e.g., ‘sad’, ‘upset’)\n"
    "5) anger: hostility, conflict, or frustration (e.g., ‘angry’, ‘irritated’)\n"
    "6) fear: anxiety, worry, or strong concern (e.g., ‘afraid’, ‘scared’)\n"
    "7) disgust: repulsion or aversion — extremely rare in this dataset.\n\n"
    "---------- CONTEXT & CHALLENGES ----------\n"
    "- The dataset is highly **imbalanced**; the majority are 'neutral' or 'happiness.'\n"
    "- Rare classes include fear, disgust, anger, sadness, and surprise.\n"
    "- English translations may soften or alter original Arabic emotion.\n"
    "- **Avoid** defaulting to 'neutral' unless you find absolutely no emotional indicators.\n\n"
    "---------- EMOTION CUES & EXAMPLES ----------\n"
    "• **happiness**: Words like 'happy', 'excited', 'love', 'wonderful', 'amazing', 'fun', or obvious positive tone.\n"
    "• **surprise**: Expressions of shock, astonishment: 'wow', 'didn't expect', 'shocked', 'unbelievable'.\n"
    "• **sadness**: Words like 'sad', 'unhappy', 'upset', 'down', 'disappointed', or negative reflection.\n"
    "• **anger**: Strong conflict or hostility: 'angry', 'mad', 'irritated', 'frustrated', scolding language.\n"
    "• **fear**: Anxiety, worry, or 'I'm afraid', 'scared', 'nervous', 'terrified'.\n"
    "• **disgust**: Repulsion, 'disgusting', 'gross', 'can't stand this'. Rare.\n"
    "• **neutral**: Trivial or factual statements, small talk, or any line lacking emotional intensity.\n\n"
    "---------- FEW-SHOT EXAMPLES ----------\n"
    "1) 'I love how you think—it's so fascinating!'\n"
    "   => Key signals: 'love', 'fascinating' => happiness\n\n"
    "2) 'I didn't expect such a direct question... Wow!'\n"
    "   => Key signals: 'didn't expect', 'wow' => surprise\n\n"
    "3) 'I'm nervous about this entire situation; it worries me.'\n"
    "   => Key signals: 'nervous', 'worries' => fear\n\n"
    "4) 'Honestly, this is making me upset—it's just not okay.'\n"
    "   => Key signals: 'upset', negative context => sadness (if there's sorrow)\n"
    "   => Could be anger if there's hostility. But if it leans more sorrowful => sadness.\n\n"
    "5) 'Stop interrupting me, I'm fed up with this!'\n"
    "   => Key signals: 'Stop interrupting', 'fed up' => anger\n\n"
    "6) 'That's honestly disgusting—I can't stand it.'\n"
    "   => Key signals: 'disgusting', 'can't stand' => disgust\n\n"
    "7) 'So, do you have any siblings?'\n"
    "   => Purely factual => neutral\n\n"
    "• neutral: “To know that I am giving up on ordinary things”\n"
    "• happiness: “They will decide if you will go back to another or not”\n"
    "• surprise: “Why do you hear and respond to you as you talk?”\n"
    "• sadness: “No, no”\n"
    "• anger: “Because it is an interrupted and flounder words”\n"
    "• fear: “You do not see me, he will be shot”\n"
    "(Note: no ‘disgust’ example is found in this dataset.)\n\n"
    "SENTENCE:\n"
    "\"{}\"\n\n"
    "YOUR TASK:\n"
    "2) Output a single label from [neutral, happiness, surprise, sadness, anger, fear, disgust].\n"
    "Remember: DO NOT default to 'neutral' if there's any emotional hint.\n"
)


predictions_baseline = []

# Wrap your list of sentences with tqdm for a progress bar
for sent in tqdm(sentences, desc="Predicting Baseline Emotions"):
    prompt = baseline_prompt_template.format(sent)
    pred = query_llm(prompt)  
    
    pred_cleaned = pred.lower()
    possible_labels = ["neutral", "happiness", "surprise", "sadness", "anger", "fear", "disgust"]
    
    found_label = "neutral"  # default to neutral if none found
    for lbl in possible_labels:
        if lbl in pred_cleaned:
            found_label = lbl
            break
    
    predictions_baseline.append(found_label)

Predicting Baseline Emotions: 100%|██████████| 982/982 [4:06:21<00:00, 15.05s/it]   


In [60]:
# -----------------------------
# 3. Compare with Gold Labels
# -----------------------------

from sklearn.metrics import f1_score


accuracy_baseline = accuracy_score(gold_labels, predictions_baseline)
conf_mat_baseline = confusion_matrix(gold_labels, predictions_baseline, labels=possible_labels)
f1_weighted = f1_score(gold_labels, predictions_baseline, average='weighted')

print("=== Baseline Prompt Evaluation ===")
print("predictions_baseline:", predictions_baseline)
print("Gold Labels:", gold_labels)
print(f"Accuracy: {accuracy_baseline:.3f}")
print(f"F1 Score: {f1_weighted:.3f}")    
print("Confusion Matrix (rows=gold, cols=pred):")
print(pd.DataFrame(conf_mat_baseline, index=possible_labels, columns=possible_labels))

=== Baseline Prompt Evaluation ===
predictions_baseline: ['neutral', 'neutral', 'surprise', 'anger', 'surprise', 'surprise', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'happiness', 'neutral', 'neutral', 'neutral', 'surprise', 'happiness', 'happiness', 'neutral', 'neutral', 'happiness', 'neutral', 'neutral', 'happiness', 'neutral', 'neutral', 'neutral', 'neutral', 'happiness', 'surprise', 'happiness', 'neutral', 'happiness', 'neutral', 'happiness', 'neutral', 'neutral', 'surprise', 'neutral', 'neutral', 'neutral', 'neutral', 'surprise', 'neutral', 'neutral', 'happiness', 'neutral', 'neutral', 'happiness', 'happiness', 'happiness', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'surprise', 'happiness', 'neutral', 'neutral', 'neutral', 'neutral', 'happiness', 'neutral', 'surprise', 'happiness', 'neutral', 'happiness', 'neutral',

---------------